<a href="https://colab.research.google.com/github/deekshdechamma/DL_skill_developer/blob/main/dl7.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Task 7: ResNet-50 Style Residual Bottleneck Block and Grouped
Convolutions
● Objective: Master deep architectural scaling paradigms by constructing bottleneck
projection networks and multi-channel grouped/depthwise separable filters.
● Required Tech Stack: PyTorch, TorchVision, PyTorch Profiler.
● Task Description: Students must build a custom ResNet-50 bottleneck block featuring
depthwise separable convolutions and learnable skip projection layers. They will run
PyTorch Profiler to analyze and optimize memory footprints, parameter counts, and
floating-point execution steps (FLOPs) under different channel scaling factors

In [1]:
#Step 1: Import libraries
import torch
import torch.nn as nn
import torchvision
from torch.profiler import profile, record_function, ProfilerActivity

print("PyTorch version:", torch.__version__)
print("TorchVision version:", torchvision.__version__)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

PyTorch version: 2.11.0+cpu
TorchVision version: 0.26.0+cpu
Device: cpu


In [2]:
#Step 2: Create the custom bottleneck block
#Input → 1×1 Conv → Depthwise 3×3 Conv → 1×1 Conv → Add Skip Connection → ReLU

class CustomBottleneck(nn.Module):

    expansion = 4

    def __init__(self, in_channels, bottleneck_channels, stride=1):
        super(CustomBottleneck, self).__init__()

        out_channels = bottleneck_channels * self.expansion

        # 1x1 convolution - reduce channels
        self.conv1 = nn.Conv2d(
            in_channels,
            bottleneck_channels,
            kernel_size=1,
            bias=False
        )

        self.bn1 = nn.BatchNorm2d(bottleneck_channels)

        # Depthwise 3x3 convolution
        self.conv2 = nn.Conv2d(
            bottleneck_channels,
            bottleneck_channels,
            kernel_size=3,
            stride=stride,
            padding=1,
            groups=bottleneck_channels,
            bias=False
        )

        self.bn2 = nn.BatchNorm2d(bottleneck_channels)

        # 1x1 convolution - expand channels
        self.conv3 = nn.Conv2d(
            bottleneck_channels,
            out_channels,
            kernel_size=1,
            bias=False
        )

        self.bn3 = nn.BatchNorm2d(out_channels)

        self.relu = nn.ReLU(inplace=True)

        # Learnable skip projection
        if stride != 1 or in_channels != out_channels:

            self.skip = nn.Sequential(

                nn.Conv2d(
                    in_channels,
                    out_channels,
                    kernel_size=1,
                    stride=stride,
                    bias=False
                ),

                nn.BatchNorm2d(out_channels)
            )

        else:
            self.skip = nn.Identity()

    def forward(self, x):

        identity = self.skip(x)

        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)

        out = self.conv2(out)
        out = self.bn2(out)
        out = self.relu(out)

        out = self.conv3(out)
        out = self.bn3(out)

        # Residual connection
        out = out + identity

        out = self.relu(out)

        return out

In [3]:
#Step 3: Test the bottleneck block
x = torch.randn(1, 64, 56, 56).to(device)

block = CustomBottleneck(
    in_channels=64,
    bottleneck_channels=64
).to(device)

output = block(x)

print("Input shape :", x.shape)
print("Output shape:", output.shape)

Input shape : torch.Size([1, 64, 56, 56])
Output shape: torch.Size([1, 256, 56, 56])


In [4]:
#Step 4: Calculate parameter count
def count_parameters(model):

    total_params = sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )

    return total_params


params = count_parameters(block)

print("Trainable Parameters:", params)

Trainable Parameters: 38720


In [5]:
print(f"Parameters: {params:,}")
print(f"Parameters (Millions): {params / 1e6:.4f} M")

Parameters: 38,720
Parameters (Millions): 0.0387 M


In [6]:
#Step 5: Check depthwise/grouped convolution
print("Input channels :", block.conv2.in_channels)
print("Output channels:", block.conv2.out_channels)
print("Groups         :", block.conv2.groups)

Input channels : 64
Output channels: 64
Groups         : 64


In [7]:
#Step 6: Standard grouped convolution example
group_conv = nn.Conv2d(
    in_channels=64,
    out_channels=128,
    kernel_size=3,
    padding=1,
    groups=4
)

group_input = torch.randn(1, 64, 56, 56)

group_output = group_conv(group_input)

print("Input shape :", group_input.shape)
print("Output shape:", group_output.shape)
print("Groups:", group_conv.groups)

Input shape : torch.Size([1, 64, 56, 56])
Output shape: torch.Size([1, 128, 56, 56])
Groups: 4


In [8]:
#Step 7: PyTorch Profiler
block.eval()

with profile(
    activities=[
        ProfilerActivity.CPU
    ] + ([ProfilerActivity.CUDA] if torch.cuda.is_available() else []),
    record_shapes=True,
    profile_memory=True,
    with_flops=True
) as prof:

    with record_function("custom_resnet_bottleneck"):

        with torch.no_grad():
            output = block(x)

print(
    prof.key_averages().table(
        sort_by="cpu_time_total",
        row_limit=15
    )
)

--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
                            Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg       CPU Mem  Self CPU Mem    # of Calls  Total KFLOPs  
--------------------------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  ------------  
        custom_resnet_bottleneck        30.70%      12.956ms       100.00%      42.200ms      42.200ms       3.06 MB     -15.31 MB             1            --  
                    aten::conv2d         0.24%     100.652us        36.76%      15.512ms       3.878ms       7.66 MB           0 B             4    234823.680  
               aten::convolution         0.21%      88.251us        36.52%      15.412ms       3.853ms       7.66 MB           0 B             4            --  
              aten::_convolution  

/usr/local/lib/python3.12/dist-packages/torch/profiler/profiler.py:224: UserWarning: Warning: Profiler clears events at the end of each cycle.Only events from the current cycle will be reported.To keep events across cycles, set acc_events=True.
  _warn_once(


In [9]:
#Step 8: Calculate total FLOPs
total_flops = 0

for event in prof.key_averages():

    if event.flops is not None:
        total_flops += event.flops

print(f"Total FLOPs: {total_flops:,}")
print(f"Total GFLOPs: {total_flops / 1e9:.4f}")

Total FLOPs: 235,626,496
Total GFLOPs: 0.2356


In [10]:
#Step 9: Test different channel scaling factors
scales = [0.5, 1.0, 1.5, 2.0]

results = []

for scale in scales:

    channels = max(1, int(64 * scale))

    model = CustomBottleneck(
        in_channels=64,
        bottleneck_channels=channels
    ).to(device)

    input_tensor = torch.randn(
        1, 64, 56, 56
    ).to(device)

    params = count_parameters(model)

    with profile(
        activities=[ProfilerActivity.CPU],
        record_shapes=True,
        profile_memory=True,
        with_flops=True
    ) as prof:

        with torch.no_grad():
            output = model(input_tensor)

    total_flops = sum(
        event.flops
        for event in prof.key_averages()
        if event.flops is not None
    )

    results.append([
        scale,
        channels,
        output.shape[1],
        params,
        total_flops
    ])

In [11]:
#Step 10: Display comparison
print(
    f"{'Scale':<10}"
    f"{'Bottleneck':<15}"
    f"{'Output Ch':<15}"
    f"{'Parameters':<15}"
    f"{'FLOPs':<20}"
)

print("-" * 75)

for r in results:

    print(
        f"{r[0]:<10}"
        f"{r[1]:<15}"
        f"{r[2]:<15}"
        f"{r[3]:<15,}"
        f"{r[4]:<20,}"
    )

Scale     Bottleneck     Output Ch      Parameters     FLOPs               
---------------------------------------------------------------------------
0.5       32             128            15,264         92,123,136          
1.0       64             256            38,720         235,626,496         
1.5       96             384            70,368         430,510,080         
2.0       128            512            110,208        676,773,888         


In [12]:
#Step 11: Memory usage
if torch.cuda.is_available():

    torch.cuda.reset_peak_memory_stats()

    model = CustomBottleneck(
        in_channels=64,
        bottleneck_channels=64
    ).cuda()

    x = torch.randn(
        1, 64, 56, 56
    ).cuda()

    with torch.no_grad():
        output = model(x)

    memory = torch.cuda.max_memory_allocated()

    print(
        "Peak GPU Memory:",
        memory / (1024 ** 2),
        "MB"
    )

else:

    print("CUDA GPU not available.")

CUDA GPU not available.


In [13]:
#Step 12: Compare normal convolution vs depthwise convolution
normal_conv = nn.Conv2d(
    64,
    64,
    kernel_size=3,
    padding=1,
    bias=False
)

depthwise_conv = nn.Conv2d(
    64,
    64,
    kernel_size=3,
    padding=1,
    groups=64,
    bias=False
)

normal_params = count_parameters(normal_conv)

depthwise_params = count_parameters(depthwise_conv)

print("Normal Conv Parameters:")
print(normal_params)

print("\nDepthwise Conv Parameters:")
print(depthwise_params)

print(
    "\nParameter Reduction:",
    normal_params / depthwise_params,
    "times"
)

Normal Conv Parameters:
36864

Depthwise Conv Parameters:
576

Parameter Reduction: 64.0 times


In [14]:
#Step 13: Full experiment in one code cell
import torch
import torch.nn as nn
from torch.profiler import profile, ProfilerActivity


device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)


def count_parameters(model):

    return sum(
        p.numel()
        for p in model.parameters()
        if p.requires_grad
    )


scales = [0.5, 1.0, 1.5, 2.0]

print("Device:", device)

print(
    f"\n{'Scale':<10}"
    f"{'Channels':<12}"
    f"{'Output':<12}"
    f"{'Params':<15}"
    f"{'GFLOPs':<15}"
)

print("-" * 64)


for scale in scales:

    channels = max(1, int(64 * scale))

    model = CustomBottleneck(
        in_channels=64,
        bottleneck_channels=channels
    ).to(device)

    x = torch.randn(
        1,
        64,
        56,
        56,
        device=device
    )

    params = count_parameters(model)

    activities = [ProfilerActivity.CPU]

    if torch.cuda.is_available():
        activities.append(ProfilerActivity.CUDA)

    with profile(
        activities=activities,
        record_shapes=True,
        profile_memory=True,
        with_flops=True
    ) as prof:

        with torch.no_grad():
            y = model(x)

    total_flops = sum(
        event.flops
        for event in prof.key_averages()
        if event.flops is not None
    )

    print(
        f"{scale:<10}"
        f"{channels:<12}"
        f"{y.shape[1]:<12}"
        f"{params:<15,}"
        f"{total_flops / 1e9:<15.4f}"
    )

Device: cpu

Scale     Channels    Output      Params         GFLOPs         
----------------------------------------------------------------
0.5       32          128         15,264         0.0921         
1.0       64          256         38,720         0.2356         
1.5       96          384         70,368         0.4305         
2.0       128         512         110,208        0.6768         
